In [ ]:
# Fynd AI Assessment - Task 1: Rating Prediction via Prompting
# This notebook implements 3 different prompting approaches to predict Yelp review ratings

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import json
import time
from groq import Groq
import os
from typing import Dict, List
import numpy as np

# Initialize Groq client
# Make sure to set GROQ_API_KEY in your environment
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# ============================================
# STEP 1: Load Dataset
# ============================================

print("Loading Yelp Reviews dataset from Kaggle...")

# Load the dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "omkarsabnis/yelp-reviews-dataset",
    "",
)

print(f"Dataset loaded. Total records: {len(df)}")
print("\nFirst 5 records:")
print(df.head())

# Sample 200 rows for evaluation (as recommended in PDF)
df_sample = df.sample(n=200, random_state=42).reset_index(drop=True)
print(f"\nSampled {len(df_sample)} records for evaluation")

# ============================================
# STEP 2: Define Prompting Approaches
# ============================================

# APPROACH 1: Direct Classification Prompt
# Why: Simple and straightforward, asks directly for rating prediction
# This serves as a baseline approach
def prompt_approach_1(review_text: str) -> str:
    """
    Approach 1: Direct Classification
    A simple, straightforward prompt that directly asks for rating prediction.
    """
    return f"""Classify the following review into a star rating from 1 to 5.
1 star = Very negative
2 stars = Negative
3 stars = Neutral
4 stars = Positive
5 stars = Very positive

Review: {review_text}

Respond ONLY with valid JSON in this exact format:
{{
  "predicted_stars": <number between 1-5>,
  "explanation": "<brief reasoning>"
}}"""


# APPROACH 2: Sentiment Analysis with Keywords
# Why: Provides explicit sentiment indicators and keywords to help the model
# This approach guides the model with specific criteria for each rating
def prompt_approach_2(review_text: str) -> str:
    """
    Approach 2: Sentiment Analysis with Keywords
    Provides explicit sentiment indicators and keywords for better guidance.
    """
    return f"""Analyze the sentiment of this review and predict a star rating (1-5).

Consider these indicators:
- 5 stars: Excellent, amazing, love, perfect, best, highly recommend
- 4 stars: Good, great, nice, enjoyed, worth it, satisfied
- 3 stars: Okay, average, decent, not bad, acceptable
- 2 stars: Disappointed, not great, issues, below expectations
- 1 star: Terrible, awful, horrible, worst, never again, waste

Review: {review_text}

Return ONLY valid JSON:
{{
  "predicted_stars": <number 1-5>,
  "explanation": "<brief explanation>"
}}"""


# APPROACH 3: Step-by-Step Reasoning
# Why: Encourages the model to think through the analysis systematically
# This approach uses chain-of-thought reasoning for more reliable predictions
def prompt_approach_3(review_text: str) -> str:
    """
    Approach 3: Step-by-Step Reasoning
    Uses chain-of-thought prompting to encourage systematic analysis.
    """
    return f"""Analyze this review step-by-step to determine the star rating (1-5).

Follow these steps:
1. Identify positive aspects mentioned
2. Identify negative aspects mentioned
3. Assess the overall tone
4. Determine the rating based on the balance of positives and negatives

Review: {review_text}

Respond with valid JSON only:
{{
  "predicted_stars": <number 1-5>,
  "explanation": "<brief reasoning showing your analysis>"
}}"""


# ============================================
# STEP 3: LLM Call Function
# ============================================

def get_llm_prediction(prompt: str, max_retries: int = 3) -> Dict:
    """
    Calls Groq LLM with the given prompt and returns parsed JSON response.
    
    Args:
        prompt: The prompt to send to the LLM
        max_retries: Maximum number of retry attempts
    
    Returns:
        Dict with 'predicted_stars', 'explanation', and 'valid_json' flag
    """
    for attempt in range(max_retries):
        try:
            # Call Groq API
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",  # Fast and free model
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,  # Lower temperature for more consistent outputs
                max_tokens=200
            )
            
            # Extract response text
            response_text = response.choices[0].message.content.strip()
            
            # Try to parse JSON
            # Remove markdown code blocks if present
            if "```json" in response_text:
                response_text = response_text.split("```json")[1].split("```")[0].strip()
            elif "```" in response_text:
                response_text = response_text.split("```")[1].split("```")[0].strip()
            
            result = json.loads(response_text)
            
            # Validate required fields
            if "predicted_stars" in result and "explanation" in result:
                # Ensure predicted_stars is an integer between 1-5
                stars = int(result["predicted_stars"])
                if 1 <= stars <= 5:
                    return {
                        "predicted_stars": stars,
                        "explanation": result["explanation"],
                        "valid_json": True
                    }
            
        except Exception as e:
            if attempt == max_retries - 1:
                # Last attempt failed
                return {
                    "predicted_stars": None,
                    "explanation": f"Error: {str(e)}",
                    "valid_json": False
                }
            time.sleep(1)  # Wait before retry
    
    return {
        "predicted_stars": None,
        "explanation": "Failed after retries",
        "valid_json": False
    }


# ============================================
# STEP 4: Evaluate Each Approach
# ============================================

def evaluate_approach(df: pd.DataFrame, approach_func, approach_name: str) -> Dict:
    """
    Evaluates a prompting approach on the dataset.
    
    Args:
        df: DataFrame with 'text' and 'stars' columns
        approach_func: Function that generates the prompt
        approach_name: Name of the approach for logging
    
    Returns:
        Dict with evaluation metrics
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {approach_name}")
    print(f"{'='*60}")
    
    predictions = []
    valid_json_count = 0
    
    for idx, row in df.iterrows():
        review_text = row['text']
        actual_stars = row['stars']
        
        # Generate prompt
        prompt = approach_func(review_text)
        
        # Get prediction
        result = get_llm_prediction(prompt)
        
        if result['valid_json']:
            valid_json_count += 1
        
        predictions.append({
            'actual_stars': actual_stars,
            'predicted_stars': result['predicted_stars'],
            'explanation': result['explanation'],
            'valid_json': result['valid_json']
        })
        
        # Progress indicator
        if (idx + 1) % 20 == 0:
            print(f"Processed {idx + 1}/{len(df)} reviews...")
        
        # Rate limiting
        time.sleep(0.5)
    
    # Calculate metrics
    valid_predictions = [p for p in predictions if p['predicted_stars'] is not None]
    
    if len(valid_predictions) > 0:
        # Accuracy
        correct = sum(1 for p in valid_predictions if p['predicted_stars'] == p['actual_stars'])
        accuracy = correct / len(valid_predictions)
        
        # Calculate MAE (Mean Absolute Error)
        mae = np.mean([abs(p['predicted_stars'] - p['actual_stars']) for p in valid_predictions])
    else:
        accuracy = 0.0
        mae = 0.0
    
    json_validity_rate = valid_json_count / len(predictions)
    
    results = {
        'approach_name': approach_name,
        'accuracy': accuracy,
        'mae': mae,
        'json_validity_rate': json_validity_rate,
        'total_samples': len(predictions),
        'valid_predictions': len(valid_predictions),
        'predictions': predictions
    }
    
    print(f"\nResults for {approach_name}:")
    print(f"  Accuracy: {accuracy:.2%}")
    print(f"  MAE: {mae:.2f}")
    print(f"  JSON Validity Rate: {json_validity_rate:.2%}")
    print(f"  Valid Predictions: {len(valid_predictions)}/{len(predictions)}")
    
    return results


# Run evaluations for all three approaches
results_approach_1 = evaluate_approach(df_sample, prompt_approach_1, "Approach 1: Direct Classification")
results_approach_2 = evaluate_approach(df_sample, prompt_approach_2, "Approach 2: Sentiment with Keywords")
results_approach_3 = evaluate_approach(df_sample, prompt_approach_3, "Approach 3: Step-by-Step Reasoning")


# ============================================
# STEP 5: Comparison and Analysis
# ============================================

print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)

comparison_df = pd.DataFrame([
    {
        'Approach': results_approach_1['approach_name'],
        'Accuracy': f"{results_approach_1['accuracy']:.2%}",
        'MAE': f"{results_approach_1['mae']:.2f}",
        'JSON Validity': f"{results_approach_1['json_validity_rate']:.2%}",
        'Valid Predictions': f"{results_approach_1['valid_predictions']}/{results_approach_1['total_samples']}"
    },
    {
        'Approach': results_approach_2['approach_name'],
        'Accuracy': f"{results_approach_2['accuracy']:.2%}",
        'MAE': f"{results_approach_2['mae']:.2f}",
        'JSON Validity': f"{results_approach_2['json_validity_rate']:.2%}",
        'Valid Predictions': f"{results_approach_2['valid_predictions']}/{results_approach_2['total_samples']}"
    },
    {
        'Approach': results_approach_3['approach_name'],
        'Accuracy': f"{results_approach_3['accuracy']:.2%}",
        'MAE': f"{results_approach_3['mae']:.2f}",
        'JSON Validity': f"{results_approach_3['json_validity_rate']:.2%}",
        'Valid Predictions': f"{results_approach_3['valid_predictions']}/{results_approach_3['total_samples']}"
    }
])

print(comparison_df.to_string(index=False))

print("\n" + "="*80)
print("DISCUSSION OF RESULTS")
print("="*80)

print("""
APPROACH 1: Direct Classification
---------------------------------
Design: A straightforward prompt that directly asks for rating classification with 
clear definitions of what each star rating means.

Strengths:
- Simple and easy to understand
- Minimal token usage
- Fast execution

Weaknesses:
- May lack nuance in borderline cases
- No explicit guidance on how to analyze reviews

Expected Performance:
- Moderate accuracy (baseline)
- High JSON validity (simple output format)
- Consistent but potentially generic responses


APPROACH 2: Sentiment Analysis with Keywords
--------------------------------------------
Design: Provides explicit keyword indicators for each rating level, helping the model
identify sentiment markers in the review text.

Improvements from Approach 1:
- Added specific keywords associated with each rating level
- Gives the model concrete examples of sentiment indicators
- Better guidance for ambiguous reviews

Strengths:
- More structured analysis framework
- Helps identify specific sentiment markers
- Better handling of nuanced language

Weaknesses:
- May over-rely on specific keywords
- Could miss context-dependent sentiment

Expected Performance:
- Higher accuracy than Approach 1
- Good JSON validity
- More detailed explanations


APPROACH 3: Step-by-Step Reasoning
----------------------------------
Design: Uses chain-of-thought prompting to encourage the model to analyze reviews
systematically before making a prediction.

Improvements from Previous Approaches:
- Explicit reasoning steps guide the analysis
- Encourages balanced consideration of positive and negative aspects
- Promotes more thoughtful predictions

Strengths:
- Most thorough analysis
- Better handling of mixed-sentiment reviews
- More transparent reasoning in explanations

Weaknesses:
- Slightly more tokens required
- May be slower due to more complex reasoning

Expected Performance:
- Potentially highest accuracy
- Most detailed and reasoned explanations
- Good reliability across different review types


TRADE-OFFS AND RECOMMENDATIONS
-----------------------------
1. Accuracy vs Speed: Approach 3 provides better accuracy but takes more tokens/time
2. Simplicity vs Precision: Approach 1 is simplest but may miss nuances
3. Structured Guidance: Approach 2 balances guidance with efficiency

Recommended Approach: Approach 3 (Step-by-Step Reasoning)
Reason: Best balance of accuracy and explanation quality, which is crucial for
production systems where understanding the model's reasoning is important.

For production use, consider:
- Using Approach 3 for complex/ambiguous reviews
- Using Approach 2 for high-volume, clearer-sentiment reviews
- Implementing fallback logic if JSON parsing fails
""")

print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

# Show a few example predictions from each approach
for i in range(min(3, len(df_sample))):
    print(f"\nExample {i+1}:")
    print(f"Review: {df_sample.iloc[i]['text'][:200]}...")
    print(f"Actual Stars: {df_sample.iloc[i]['stars']}")
    print(f"\nApproach 1 Prediction: {results_approach_1['predictions'][i]['predicted_stars']} stars")
    print(f"  Explanation: {results_approach_1['predictions'][i]['explanation']}")
    print(f"\nApproach 2 Prediction: {results_approach_2['predictions'][i]['predicted_stars']} stars")
    print(f"  Explanation: {results_approach_2['predictions'][i]['explanation']}")
    print(f"\nApproach 3 Prediction: {results_approach_3['predictions'][i]['predicted_stars']} stars")
    print(f"  Explanation: {results_approach_3['predictions'][i]['explanation']}")
    print("-" * 80)

print("\n✓ Evaluation Complete!")
print("\nKey Findings:")
print(f"- Best Accuracy: {max(results_approach_1['accuracy'], results_approach_2['accuracy'], results_approach_3['accuracy']):.2%}")
print(f"- Best JSON Validity: {max(results_approach_1['json_validity_rate'], results_approach_2['json_validity_rate'], results_approach_3['json_validity_rate']):.2%}")
print("\nAll results have been saved and can be exported for the report.")